# 🚀 Huấn Luyện VSL Alphabet (Level 1 - Fingerspelling) Trên Cloud GPU

**Dự án:** Vietnamese Sign Language Translator (VSLT)  
**Phân hệ:** Cấp 1 - Bảng Chữ Cái Ký Hiệu (Fingerspelling)  
**Dữ liệu:** VSL Alphabet Pilot Dataset (1,875 clips, 15 signers, 25 lớp ký hiệu chuẩn tiếng Việt).  
**Thiết kế thử nghiệm:** Signer-Disjoint Split (10 train signers, 2 val signers, 3 test signers).  

---

### 📌 Mục Tiêu Thử Nghiệm:
1. Huấn luyện 2 kiến trúc baseline: **Static MLP** (21 landmarks palm-scale normalized, 63 chiều) và **Temporal BiGRU** (chuỗi 30 frames).
2. Đánh giá tính tổng quát trên người ra ký hiệu hoàn toàn mới (Signers `S03, S14, S15`).
3. Xuất file checkpoint chuẩn `alphabet_best.pt` để tích hợp vào hệ thống suy luận thời gian thực.

In [ ]:
# =============================================================
# 1. KIỂM TRA MÔI TRƯỜNG VÀ GPU
# =============================================================
import os
import sys
import torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')

print('=' * 60)
print('MÔI TRƯỜNG HUẤN LUYỆN:')
print(f'  Google Colab: {IN_COLAB}')
print(f'  Kaggle:       {IN_KAGGLE}')
print(f'  CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:          {torch.cuda.get_device_name(0)}')
print('=' * 60)


In [ ]:
# =============================================================
# 2. GIẢI NÉN VÀ KIỂM TRA DỮ LIỆU
# =============================================================
import zipfile
from pathlib import Path

zip_path = Path('data/vsl_alphabet_cloud_data.zip')
if not zip_path.exists():
    # Kaggle input dataset fallback
    kaggle_paths = list(Path('/kaggle/input').rglob('vsl_alphabet_cloud_data.zip')) if Path('/kaggle/input').exists() else []
    if kaggle_paths:
        zip_path = kaggle_paths[0]
    else:
        print('Vui lòng upload file vsl_alphabet_cloud_data.zip lên thư mục làm việc.')

if zip_path.exists():
    print(f'Đang giải nén {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('.')
    print('Giải nén hoàn tất!')

print(f'Train split tồn tại: {Path("data/vsl_alphabet_pilot/splits/train.csv").exists()}')
print(f'Val split tồn tại:   {Path("data/vsl_alphabet_pilot/splits/val.csv").exists()}')
print(f'Test split tồn tại:  {Path("data/vsl_alphabet_pilot/splits/test.csv").exists()}')


In [ ]:
# =============================================================
# 3. HUẤN LUYỆN BASELINE 1: STATIC MLP (63 DIMS)
# =============================================================
from train_alphabet import run_training

print('Bắt đầu huấn luyện Static MLP...')
mlp_res = run_training(
    model_type='mlp',
    epochs=35,
    batch_size=32,
    lr=1e-3,
    save_path='checkpoints/alphabet_mlp_best.pt'
)


In [ ]:
# =============================================================
# 4. HUẤN LUYỆN BASELINE 2: TEMPORAL BIGRU (CHUỖI 30 FRAMES)
# =============================================================
print('Bắt đầu huấn luyện Temporal BiGRU...')
bigru_res = run_training(
    model_type='bigru',
    epochs=35,
    batch_size=32,
    lr=8e-4,
    save_path='checkpoints/alphabet_bigru_best.pt'
)


In [ ]:
# =============================================================
# 5. SO SÁNH HIỆU NĂNG VÀ CHỌN CHECKPOINT TỐT NHẤT
# =============================================================
import shutil

print('=' * 60)
print('KẾT QUẢ SO SÁNH TRÊN TEST SET (SIGNER DISJOINT):')
print(f'  Static MLP:     Top-1 = {mlp_res["test_top1"]*100:.2f}% | Top-3 = {mlp_res["test_top3"]*100:.2f}%')
print(f'  Temporal BiGRU: Top-1 = {bigru_res["test_top1"]*100:.2f}% | Top-3 = {bigru_res["test_top3"]*100:.2f}%')
print('=' * 60)

if mlp_res['test_top1'] >= bigru_res['test_top1']:
    winner = 'Static MLP'
    shutil.copy('checkpoints/alphabet_mlp_best.pt', 'checkpoints/alphabet_best.pt')
else:
    winner = 'Temporal BiGRU'
    shutil.copy('checkpoints/alphabet_bigru_best.pt', 'checkpoints/alphabet_best.pt')

print(f'Mô hình chiến thắng: {winner} -> Đã lưu vào checkpoints/alphabet_best.pt')
if IN_COLAB:
    from google.colab import files
    files.download('checkpoints/alphabet_best.pt')
